# Projet d'analyse de corpus — Prétraitement de *Le Féminisme* d'Émile Faguet

**Objectif : étudier la dépendance des traitements lexicaux au nettoyage préalable du corpus.**

Une mesure lexicale repose sur les unités reconnues dans le texte. La préparation de ces unités (délimitation, normalisation, traitement des éléments éditoriaux) conditionne donc les résultats obtenus. Voir Manning, Raghavan et Schütze (2008, chap. 2) et Bird, Klein et Loper (2009, chap. 3).

**Déroulement :** découvrir les fonctions ; produire et sauvegarder une première version nettoyée ; inspecter le document et rechercher d'éventuelles anomalies ; estimer leur fréquence et décider de poursuivre ou de tester une révision ; comparer enfin les mesures lexicales.

Les sources sont communes : texte original figé, fonctions Python issues d'une révision Git identifiée et profil de nettoyage partagé. Vos observations et décisions restent personnelles.

Enregistrez une copie de ce notebook dans **votre Drive**. En fin de séance, déposez les résultats dans le **dossier partagé**.

*Références :* Manning, C. D., Raghavan, P. et Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press, chapitre 2, « The term vocabulary and postings lists ». https://doi.org/10.1017/CBO9780511809071 ; Bird, S., Klein, E. et Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media, chapitre 3. https://www.nltk.org/book/ch03.html.

## Initialisation — Versions et sources

Deux paramètres servent à reprendre cette analyse après une mise à jour : la **révision Git** des fonctions et la **version du notebook**. Pour évaluer une nouvelle version des fonctions, créez une copie du notebook, changez la révision Git, augmentez la version du notebook et choisissez le mode `exploration`.

Le mode `reference` contrôle la concordance avec le nettoyage de référence ; le mode `exploration` conserve les mêmes règles générales mais autorise l'examen d'une sortie modifiée. Les résultats d'exploration seront identifiés comme tels.

In [ ]:
import hashlib
import importlib
import json
from pathlib import Path
import subprocess
import sys
from urllib.parse import quote
from urllib.request import urlopen

DEPOT_LIB = "dreymond732/tal-notebook-evaluator"
SOUS_DOSSIER_LIB = "PT-LLMvsPy"

# Paramètres de version : créez une copie du notebook avant de les modifier.
VERSION_NOTEBOOK = "2.0.0"
REVISION_GIT = "main"  # Pour tester la branche avant fusion : "feat/pt-llmvspy-colab-integrated"
MODE_REVISION = "reference"  # "reference" ou "exploration"

assert MODE_REVISION in {"reference", "exploration"}
DOSSIER_DRIVE_ID = "18KPGO5jGPt5f8t_Gq1km79hno4rqy1xO"
URL_DOSSIER_DRIVE = "https://drive.google.com/drive/folders/" + DOSSIER_DRIVE_ID
DOSSIER_TRAVAIL = Path.cwd() / ("analyse_faguet_v" + VERSION_NOTEBOOK.replace(".", "_"))
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)

try:
    import google.colab
    EST_COLAB = True
except ImportError:
    EST_COLAB = False

print("Notebook :", VERSION_NOTEBOOK)
def figer_revision_git(depot, reference):
    """Résout une branche ou un tag en commit immuable pour toute la session."""
    url = ("https://api.github.com/repos/" + depot
           + "/commits/" + quote(reference, safe=""))
    with urlopen(url, timeout=30) as reponse:
        commit = json.loads(reponse.read().decode("utf-8"))["sha"]
    if len(commit) != 40 or any(c not in "0123456789abcdef" for c in commit):
        raise ValueError("Révision Git incorrecte")
    return commit

REVISION_GIT_RESOLUE = figer_revision_git(DEPOT_LIB, REVISION_GIT)
print("Révision demandée :", REVISION_GIT)
print("Commit des fonctions :", REVISION_GIT_RESOLUE)
print("Mode :", MODE_REVISION)

In [ ]:
def installer_bibliotheque_depuis_git():
    for racine in (Path.cwd(), Path.cwd().parent):
        if (racine / "src" / "tal_toolkit" / "inspection_td01.py").exists():
            sys.path.insert(0, str(racine / "src"))
            print("Bibliothèque chargée depuis :", racine)
            return "clone-local"
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps",
        f"git+https://github.com/{DEPOT_LIB}.git@{REVISION_GIT_RESOLUE}"
        f"#subdirectory={SOUS_DOSSIER_LIB}",
    ])
    importlib.invalidate_caches()
    return REVISION_GIT_RESOLUE

REFERENCE_BIBLIOTHEQUE = installer_bibliotheque_depuis_git()

try:
    import spacy
    from packaging.version import Version
    if Version(spacy.__version__) < Version("3.8"):
        raise ImportError("spaCy ancien")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "spacy>=3.8,<4"
    ])
    import spacy

from tal_toolkit import (
    __version__ as VERSION_BIBLIOTHEQUE, charger_corpus,
    OperationPretraitement, PlanPretraitement,
    executer_pretraitements, recenser_elements, composer_texte_epure,
    exporter_texte_approuve, comparer_corpus_td01,
    preparer_archive_td01, deposer_archive_drive_api,
    preparer_faguet_standard_td01, plan_standard_td01,
    verifier_corpus_standard_td01, verifier_standardisation_td01,
    SHA_FAGUET_TD01, SHA_FAGUET_EPURE_TD01, METRIQUES_TD01,
    localiser_dans_original, fonctions_ayant_modifie_zone,
    compter_motif, valider_inspection,
)
assert tuple(int(x) for x in VERSION_BIBLIOTHEQUE.split(".")[:2]) >= (0, 3)
print("Bibliothèque :", VERSION_BIBLIOTHEQUE, "| spaCy :", spacy.__version__)

### Charger le texte original et le document Gemini

Le texte source est conservé pour retrouver ultérieurement toute zone concernée par le nettoyage. Le document Gemini fournit les mesures à mettre en regard des calculs.

In [ ]:
COMMIT_SOURCE = "b92e92b6c216014283056c83aa545881a447b3a5"
CHEMIN_SOURCE = "Notebooks TD/S3/ressources/faguet_source.txt"
SHA256_SOURCE = "c9832aecd99b11f3c769d11173fd6039a4948a8ee86484658dd8c7a25848a8fa"

fichier_original = DOSSIER_TRAVAIL / "faguet_source.txt"
if not fichier_original.exists():
    url = (
        "https://raw.githubusercontent.com/dreymond732/tal-notebook-evaluator/"
        + COMMIT_SOURCE + "/" + quote(CHEMIN_SOURCE, safe="/")
    )
    with urlopen(url, timeout=60) as reponse:
        fichier_original.write_bytes(reponse.read())

corpus_source = charger_corpus(fichier_original, sha256_attendu=SHA256_SOURCE)
verifier_corpus_standard_td01(corpus_source)
print("Corpus chargé :", len(corpus_source.texte), "caractères")

In [ ]:
URL_GEMINI = (
    "https://raw.githubusercontent.com/" + DEPOT_LIB
    + "/" + REVISION_GIT_RESOLUE + "/" + SOUS_DOSSIER_LIB + "/data/annonces_globales_gemini.json"
)
with urlopen(URL_GEMINI, timeout=30) as reponse:
    ANNONCES_GEMINI = json.loads(reponse.read().decode("utf-8"))
assert ANNONCES_GEMINI["version"] == "gemini_metrics_globales_v1"
for item in ANNONCES_GEMINI["annonces"]:
    print(item["libelle"], ":", item["valeur"])

## 1 — Fonctions disponibles

Le nettoyage peut porter sur les **notes et renvois éditoriaux**, les **séparateurs visuels**, les **retours de ligne typographiques** ou certains caractères. Les titres sont conservés.

| Fonction | Rôle |
|---|---|
| `recenser_elements()` | Repérer les éléments présents dans le texte |
| `executer_pretraitements()` | Appliquer le plan et compter les modifications |
| `composer_texte_epure()` | Construire le texte lisible après masquage et mise en forme |
| `exporter_texte_approuve()` | Enregistrer une version retenue et ses informations de provenance |

Le nettoyage du texte, le découpage en tokens et le choix des tokens à compter sont des opérations distinctes.

In [ ]:
inventaire = recenser_elements(corpus_source, max_exemples=12)
print("Retours CRLF :", inventaire["retours_crlf"])
for nom, donnees in inventaire["categories"].items():
    print("\n", nom, ":", donnees["nombre"])
    for exemple in donnees["exemples"][:5]:
        print("  ", exemple["debut"], repr(exemple["texte"][:100]))

def operation(nom, justification, cibles=()):
    return OperationPretraitement(nom, justification, tuple(cibles))

### Illustrations des transformations

Les exemples suivants montrent comment les fonctions réagissent aux renvois de notes, aux coupures de ligne, aux séparateurs, aux titres et aux césures. Ils constituent des tests unitaires sur des textes courts.

In [ ]:
EXEMPLES_FOURNIS = [
    {
        "nom": "retours typographiques",
        "entree": "Une phrase\r\ncontinue.\r\n\r\nAutre paragraphe.",
        "ops": [operation("recomposer_lignes", "Réunir les lignes typographiques")],
        "attendu": "Une phrase continue.\n\nAutre paragraphe.",
    },
    {
        "nom": "apostrophes courbes",
        "entree": "L’homme et l’histoire.",
        "ops": [operation("apostrophes_droites", "Normaliser les apostrophes")],
        "attendu": "L'homme et l'histoire.",
    },
    {
        "nom": "appel et contenu de note",
        "entree": "Un propos[1].\r\n\r\n[Note 1: Une référence.]\r\n\r\nLa suite.",
        "ops": [operation("masquer_appels_notes", "Masquer l'appel"),
                operation("masquer_blocs_notes", "Masquer le bloc")],
        "attendu": "Un propos.\n\nLa suite.",
    },
    {
        "nom": "séparateur d'astérisques",
        "entree": "Texte un.\r\n\r\n* * * * *\r\n\r\nTexte deux.",
        "ops": [operation("masquer_separateurs", "Masquer le séparateur")],
        "attendu": "Texte un.\n\nTexte deux.",
    },
    {
        "nom": "titre conservé",
        "entree": "IDÉES GÉNÉRALES\r\n\r\nLe chapitre commence.",
        "ops": [],
        "attendu": "IDÉES GÉNÉRALES\n\nLe chapitre commence.",
    },
    {
        "nom": "césure approuvée",
        "entree": "Une infor-\r\nmation importante.",
        "ops": [operation(
            "raccorder_cesures_validees", "Césure validée sur cet exemple",
            ("infor-\r\nmation",),
        )],
        "attendu": "Une information importante.",
    },
]
for cas in EXEMPLES_FOURNIS:
    plan_demo = PlanPretraitement(tuple(cas["ops"]), identifiant="demo-v1")
    test = executer_pretraitements(cas["entree"], plan=plan_demo)
    observe = composer_texte_epure(test).texte
    assert observe == cas["attendu"], cas["nom"]
    assert test.original == cas["entree"]
    assert test.indices_conserves
    print("OK :", cas["nom"], "=>", repr(observe))

## 2 — Nettoyer le corpus complet et sauvegarder un premier résultat

Le profil commun retire les notes et séparateurs éditoriaux, puis recompose les lignes typographiques. On obtient une première version exploitable, **sans encore décider qu'elle est satisfaisante**.

Le texte produit est enregistré sous forme de fichier `.txt` dans l'espace de travail. Ouvrez-le ensuite dans **Bloc-notes ou un éditeur de texte simple** : parcourez le début, le milieu, la fin et plusieurs zones choisies librement. Cherchez notamment des caractères résiduels, des mots mal raccordés, une ponctuation altérée ou une perte de structure (paragraphes, titres, citations).

In [ ]:
plan_global = plan_standard_td01()
if MODE_REVISION == "reference":
    resultat_global, vue_epuree, preuve_standardisation = (
        preparer_faguet_standard_td01(corpus_source)
    )
else:
    resultat_global = executer_pretraitements(
        corpus_source, plan=plan_global, max_exemples=40
    )
    vue_epuree = composer_texte_epure(resultat_global)
    preuve_standardisation = None
    assert resultat_global.sha256_original == SHA256_SOURCE

assert len(vue_epuree.indices_source) == len(vue_epuree.texte)

FICHIER_APERCU = DOSSIER_TRAVAIL / (
    "faguet_nettoye_v" + VERSION_NOTEBOOK.replace(".", "_")
    + ("_exploration" if MODE_REVISION == "exploration" else "_reference")
    + ".txt"
)
with FICHIER_APERCU.open("w", encoding="utf-8", newline="") as fichier:
    fichier.write(vue_epuree.texte)

print("Document à ouvrir :", FICHIER_APERCU)
print("Caractères : original", len(corpus_source.texte),
      "| nettoyé", len(vue_epuree.texte))
for op_effective in resultat_global.execution:
    print(op_effective["operation"], ":", op_effective["modifications"], "modifications")

In [ ]:
# Dans Colab : téléchargez le fichier texte, puis ouvrez-le avec Bloc-notes.
if EST_COLAB:
    from google.colab import files
    files.download(str(FICHIER_APERCU))
else:
    print("Ouvrez ce fichier dans votre éditeur :", FICHIER_APERCU)

## 3 — Rechercher et documenter les anomalies

Après lecture du fichier nettoyé, si vous suspectez une erreur, procédez de façon précise :

1. Choisissez dans **le texte original** une courte suite de caractères caractéristique du problème (l'**ancre**).
2. Retrouvez sa position et vérifiez la fenêtre de texte autour d'elle. Si l'ancre apparaît plusieurs fois, sélectionnez la bonne occurrence.
3. Examinez quelles fonctions ont modifié cette zone. Il s'agit de **pistes de diagnostic**, à confronter au texte.
4. Choisissez un motif représentatif et comptez ses occurrences dans le texte original et le texte nettoyé. Un motif littéral dénombre exactement cette suite de caractères ; une expression régulière peut couvrir plusieurs variantes. **Ce comptage estime la portée du motif, pas nécessairement celle de toutes les erreurs apparentées.**

Vous pouvez documenter plusieurs anomalies. Si vous n'en trouvez aucune, décrivez les parties du document réellement inspectées : un résultat sans anomalie reste une observation utile.

In [ ]:
# EXEMPLE de recherche : remplacer la chaîne par une ancre de votre choix.
ANCRE_A_CHERCHER = ""         # Suite exacte présente dans le texte original
NUMERO_OCCURRENCE = 1         # 1 pour la première occurrence, 2 pour la deuxième, etc.
RAYON_CONTEXTE = 120

if ANCRE_A_CHERCHER:
    zone = localiser_dans_original(
        corpus_source.texte, ANCRE_A_CHERCHER,
        occurrence=NUMERO_OCCURRENCE, contexte=RAYON_CONTEXTE,
    )
    print("Nombre de correspondances :", zone["nombre_correspondances"])
    print("Bornes vérifiées :", zone["fenetre_debut"], zone["fenetre_fin"])
    print("Fenêtre originale :", repr(zone["fenetre_originale"]))
    print("Vue de contrôle :", repr(resultat_global.analyse[
        zone["fenetre_debut"]:zone["fenetre_fin"]
    ]))
    print("Opérations intervenues dans la fenêtre :")
    for intervention in fonctions_ayant_modifie_zone(
        resultat_global, zone["fenetre_debut"], zone["fenetre_fin"]
    ):
        print(" ", intervention)
else:
    print("Renseignez ANCRE_A_CHERCHER pour examiner une zone du texte.")

In [ ]:
# Estimer les occurrences d'un motif lié au problème observé.
MOTIF_A_COMPTER = ""     # Mot, caractères résiduels ou expression régulière
MODE_MOTIF = "litteral"  # "litteral" ou "regex"

if MOTIF_A_COMPTER:
    resultat_occurrences = compter_motif(
        corpus_source.texte, vue_epuree.texte, MOTIF_A_COMPTER, mode=MODE_MOTIF
    )
    print("Occurrences originales :", resultat_occurrences["occurrences_original"])
    print("Occurrences après nettoyage :", resultat_occurrences["occurrences_epure"])
    print("Écart :", resultat_occurrences["difference"])
    print(resultat_occurrences["limite"])
else:
    print("Renseignez MOTIF_A_COMPTER pour évaluer la fréquence d'un cas.")

### Journal d'inspection et décision

Dans `ANOMALIES_A_EXAMINER`, décrivez chaque problème repéré : une ancre du texte original, son numéro d'occurrence, le motif utilisé pour le comptage, la fonction suspectée et votre interprétation.

La cellule suivante retrouve les positions, vérifie la fenêtre et calcule les effectifs. **N'inventez pas de contre-exemple si vous n'en avez pas trouvé.**

Enfin, décidez de **conserver** ce nettoyage ou de **réviser** les fonctions. Une anomalie rare peut être acceptée si ses conséquences sont expliquées ; une perte de structure importante peut justifier une nouvelle révision. La décision appartient à votre analyse.

In [ ]:
# Un problème = un dictionnaire ; laisser [] si aucun problème repéré.
ANOMALIES_A_EXAMINER = [
    # {
    #   "ancre": "suite exacte retrouvée dans le texte original",
    #   "occurrence": 1,
    #   "motif": "motif à dénombrer",
    #   "mode": "litteral",  # ou "regex"
    #   "fonction_suspectee": "recomposer_lignes",
    #   "commentaire": "Expliquer l'effet suspecté sur la structure du passage.",
    # },
]
METHODE_INSPECTION = ""  # Où et comment avez-vous examiné le texte nettoyé ?
DECISION_SUITE = ""      # "conserver" ou "reviser"
JUSTIFICATION_DECISION = ""  # Peser la fréquence et la gravité des problèmes.

anomalies_documentees = []
for candidat in ANOMALIES_A_EXAMINER:
    zone = localiser_dans_original(
        corpus_source.texte, candidat["ancre"],
        occurrence=int(candidat.get("occurrence", 1)), contexte=120,
    )
    estimation = compter_motif(
        corpus_source.texte, vue_epuree.texte,
        candidat["motif"], mode=candidat.get("mode", "litteral"),
    )
    fonctions = fonctions_ayant_modifie_zone(
        resultat_global, zone["fenetre_debut"], zone["fenetre_fin"]
    )
    anomalies_documentees.append({
        "ancre": candidat["ancre"],
        "debut": zone["debut"], "fin": zone["fin"],
        "fenetre_originale": zone["fenetre_originale"],
        "nombre_correspondances_ancre": zone["nombre_correspondances"],
        "fonction_suspectee": candidat["fonction_suspectee"],
        "fonctions_intervenues": fonctions,
        "commentaire": candidat["commentaire"],
        "estimation": estimation,
    })
    print("Zone :", zone["debut"], "-", zone["fin"])
    print("Fonctions possibles :", [x["operation"] for x in fonctions])
    print("Occurrences motif original/nettoyé :",
          estimation["occurrences_original"], estimation["occurrences_epure"])

journal_inspection = {
    "decision": DECISION_SUITE,
    "methode_inspection": METHODE_INSPECTION,
    "justification_decision": JUSTIFICATION_DECISION,
    "anomalies": anomalies_documentees,
    "revision_git": REVISION_GIT_RESOLUE,
    "version_notebook": VERSION_NOTEBOOK,
    "mode_revision": MODE_REVISION,
}
valider_inspection(journal_inspection)
FICHIER_INSPECTION = DOSSIER_TRAVAIL / "journal_inspection.json"
FICHIER_INSPECTION.write_text(
    json.dumps(journal_inspection, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Journal enregistré :", FICHIER_INSPECTION)
print("Anomalies relevées :", len(anomalies_documentees))
print("Décision :", DECISION_SUITE)
if DECISION_SUITE == "reviser" and EST_COLAB:
    from google.colab import files
    files.download(str(FICHIER_INSPECTION))  # Conserver le diagnostic avant de changer de version.

### Décider du nettoyage, puis poursuivre l'analyse

La décision documentée dans `journal_inspection` ne doit pas empêcher de mesurer les effets du traitement.

- **`conserver`** : le nettoyage est retenu pour cette version ; les mesures pourront accompagner un livrable validé.
- **`reviser`** : le nettoyage mérite une correction ; **poursuivez néanmoins les calculs** sur cette version provisoire. Ils permettront d'évaluer l'impact des anomalies et de comparer la version actuelle à la suivante.

Dans les deux cas, le notebook enregistrera le tableau des mesures et une fiche récapitulative précisant la décision, la source et les versions. Seul le choix `conserver` autorise l'export du texte comme version approuvée. Pour tester une correction, reprenez une copie du notebook avec `VERSION_NOTEBOOK` et `REVISION_GIT` actualisées.

In [ ]:
# La décision a déjà été décrite et vérifiée dans le journal d'inspection.
valider_inspection(journal_inspection)
assert FICHIER_APERCU.is_file(), "Exécutez d'abord le nettoyage intégral."

STATUT_ANALYSE = (
    "retenu" if DECISION_SUITE == "conserver" else "provisoire_a_reviser"
)
print("Décision :", DECISION_SUITE)
print("Statut des mesures qui suivent :", STATUT_ANALYSE)
print("Texte utilisé pour les calculs :", FICHIER_APERCU)
if DECISION_SUITE == "reviser":
    print(
        "Les mesures restent possibles : elles serviront à comparer "
        "l'effet d'une correction ultérieure."
    )

## 4 — Calculer et comparer les mesures lexicales

Les **deux décisions** (`conserver` et `reviser`) mènent à cette étape. Le texte de travail est celui sauvegardé après le nettoyage complet, sans nouvelle transformation.

Le même découpage spaCy et les mêmes règles de sélection sont appliqués au texte original et à la version préparée : tokens alphabétiques, formes minuscules, pas de lemmatisation.

Le tableau présente les **tokens**, **types**, **hapax**, **TTR**, part d'hapax, nombre de phrases et longueur moyenne, ainsi que les écarts. En cas de révision envisagée, il constitue un **point de comparaison provisoire**, non une validation du nettoyage.

In [ ]:
# Mesurer directement le fichier déjà produit : l'export approuvé n'est pas requis.
corpus_epure = charger_corpus(
    FICHIER_APERCU, sha256_attendu=vue_epuree.sha256_texte_utf8
)
nlp = spacy.blank("fr")
nlp.add_pipe("sentencizer")
nlp.max_length = max(nlp.max_length, len(corpus_source.texte) + 1000)

resultat_comparaison = comparer_corpus_td01(
    corpus_source, corpus_epure, nlp, protocole=METRIQUES_TD01
)
assert resultat_comparaison["source_originale_sha256"] == SHA256_SOURCE
assert resultat_comparaison["source_epuree_sha256"] == vue_epuree.sha256_texte_utf8

import pandas as pd
tableau_comparaison = pd.DataFrame(resultat_comparaison["comparaison"])
display(tableau_comparaison)

# Conserver la comparaison et son statut, même si une correction est demandée.
FICHIER_COMPARAISON = DOSSIER_TRAVAIL / "comparaison_metriques.csv"
tableau_comparaison.to_csv(FICHIER_COMPARAISON, index=False, encoding="utf-8")

fiche_suivi = {
    "schema": "analyse-faguet-suivi-v1",
    "decision_nettoyage": DECISION_SUITE,
    "statut": STATUT_ANALYSE,
    "revision_git": REVISION_GIT_RESOLUE,
    "version_notebook": VERSION_NOTEBOOK,
    "version_bibliotheque": VERSION_BIBLIOTHEQUE,
    "mode_revision": MODE_REVISION,
    "sha256_original": SHA256_SOURCE,
    "sha256_texte_prepare": vue_epuree.sha256_texte_utf8,
    "fichier_prepare": FICHIER_APERCU.name,
    "protocole_mesures": resultat_comparaison["protocole_commun"],
    "logiciel": resultat_comparaison["logiciel"],
    "journal_inspection": FICHIER_INSPECTION.name,
}
FICHIER_SUIVI = DOSSIER_TRAVAIL / "fiche_suivi.json"
FICHIER_SUIVI.write_text(
    json.dumps(fiche_suivi, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Tableau enregistré :", FICHIER_COMPARAISON)
print("Fiche enregistrée :", FICHIER_SUIVI)
print("Statut :", STATUT_ANALYSE)

In [ ]:
# L'export approuvé est facultatif pour calculer : il concerne seulement
# un nettoyage que vous choisissez de conserver.
export_epuration = None
manifeste_epuration = None

if DECISION_SUITE == "conserver":
    export_epuration = exporter_texte_approuve(
        resultat_global, DOSSIER_TRAVAIL / "resultats_epuration",
        decisions={
            op["nom"]: "approuve"
            for op in resultat_global.plan["operations"]
        },
        confirmation_sha256=SHA256_SOURCE,
        justification_globale=JUSTIFICATION_DECISION,
        approbation_compaction=True,
    )
    manifeste_epuration = json.loads(
        Path(export_epuration["manifeste"]).read_text(encoding="utf-8")
    )
    assert export_epuration["sha256_epure"] == vue_epuree.sha256_texte_utf8
    if MODE_REVISION == "reference":
        verifier_standardisation_td01(
            preuve_standardisation, manifeste_epuration, resultat_comparaison
        )
    print("Texte retenu :", export_epuration["texte"])
else:
    print("Nettoyage à réviser : les mesures et le journal sont enregistrés.")
    print("Aucun export n'est déclaré approuvé.")

### Interpréter et conserver les résultats

Commentez les variations des tokens, des types, des hapax et du nombre de phrases. Appuyez-vous sur votre journal d'inspection et sur les transformations réellement observées.

Un nettoyage **retenu** peut être déposé comme résultat approuvé de cette version. Un nettoyage **à réviser** donne lieu à un dossier provisoire, utile pour documenter l'anomalie et reprendre les calculs dans une nouvelle version. Les deux catégories ne sont pas confondues.

In [ ]:
MON_IDENTIFIANT = ""
BILAN_PERSONNEL = """
"""
VALIDATION_ANALYSE = False

assert MON_IDENTIFIANT.strip(), "Indiquez votre identifiant de dépôt"
assert len(BILAN_PERSONNEL.strip()) >= 50, "Rédigez votre analyse des résultats"
assert VALIDATION_ANALYSE is True, "Relisez votre analyse avant de continuer"

if DECISION_SUITE == "conserver":
    livrable = preparer_archive_td01(
        MON_IDENTIFIANT.strip(), export_epuration,
        [], resultat_comparaison, BILAN_PERSONNEL,
        DOSSIER_TRAVAIL / "depots_locaux",
        version_bibliotheque=VERSION_BIBLIOTHEQUE,
        standardisation_td01=(
            preuve_standardisation if MODE_REVISION == "reference" else None
        ),
        inspection=journal_inspection,
        revision_git=REVISION_GIT_RESOLUE,
        version_notebook=VERSION_NOTEBOOK,
        mode_revision=MODE_REVISION,
    )
else:
    # Dossier PROVISOIRE distinct de l'archive des textes approuvés.
    # Il reste téléchargeable pour la revue et la prochaine révision.
    import re
    import zipfile
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{2,49}", MON_IDENTIFIANT)
    from datetime import datetime, timezone
    date_archive = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    archive_locale = (
        DOSSIER_TRAVAIL / "depots_locaux"
        / f"FAGUET_{MON_IDENTIFIANT}_v{VERSION_NOTEBOOK.replace('.', '-')}_{date_archive}_provisoire.zip"
    )
    archive_locale.parent.mkdir(parents=True, exist_ok=True)
    if archive_locale.exists():
        raise FileExistsError(
            "Archive provisoire déjà créée : renommez-la ou augmentez VERSION_NOTEBOOK"
        )
    pieces = {
        "faguet_nettoye_provisoire.txt": FICHIER_APERCU,
        "inspection.json": FICHIER_INSPECTION,
        "comparaison_metriques.csv": FICHIER_COMPARAISON,
        "fiche_suivi.json": FICHIER_SUIVI,
    }
    with zipfile.ZipFile(archive_locale, "x", zipfile.ZIP_DEFLATED) as archive:
        for nom, chemin in pieces.items():
            archive.write(chemin, arcname=nom)
        archive.writestr(
            "rapport.md",
            "# Analyse provisoire de Faguet\n\n"
            + BILAN_PERSONNEL.strip() + "\n"
        )
    livrable = {
        "archive": str(archive_locale),
        "fichiers": sorted([*pieces, "rapport.md"]),
        "statut": "provisoire_a_reviser",
    }

print("Archive locale :", livrable["archive"])
print("Pièces :", ", ".join(livrable["fichiers"]))
print("Statut :", STATUT_ANALYSE)

## Déposer ou conserver les résultats

**Nettoyage retenu :** cochez la confirmation puis utilisez le bouton « Déposer mes résultats sur Drive ». La copie distante est vérifiée.

**Nettoyage à réviser :** une archive provisoire contient déjà le texte, le journal d'inspection, le tableau comparatif et la fiche des versions. Vous pouvez la télécharger dans la cellule suivante pour la revue ou la transmettre manuellement dans le <https://drive.google.com/drive/folders/18KPGO5jGPt5f8t_Gq1km79hno4rqy1xO>. Elle n'est pas déclarée validée.

Les calculs ne sont plus bloqués par le besoin d'une révision.

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

try:
    if EST_COLAB:
        from google.colab import output as colab_output
        colab_output.enable_custom_widget_manager()
except Exception:
    pass

confirmation_depot = widgets.Checkbox(
    value=False,
    description="J'ai relu mes résultats et j'autorise leur dépôt dans le dossier partagé",
    indent=False,
    layout=widgets.Layout(width="95%"),
)
bouton_depot = widgets.Button(
    description="Déposer mes résultats sur Drive",
    button_style="success", icon="cloud-upload",
    layout=widgets.Layout(width="320px"),
)
sortie_depot = widgets.Output()

def _deposer_td01(_):
    with sortie_depot:
        clear_output(wait=True)
        if not confirmation_depot.value:
            print("Confirmez d'abord l'envoi après avoir relu l'archive.")
            return
        bouton_depot.disabled = True
        try:
            if not EST_COLAB:
                raise RuntimeError("Le dépôt direct nécessite Google Colab")
            # Consentement Google au moment du dépôt, pas pendant les analyses.
            from google.colab import auth
            auth.authenticate_user()
            from googleapiclient.discovery import build
            # Après auth.authenticate_user(), le client utilise
            # les identifiants utilisateur disponibles dans Colab.
            service = build("drive", "v3", cache_discovery=False)
            preuve = deposer_archive_drive_api(
                livrable["archive"], service, DOSSIER_DRIVE_ID,
                confirmation_explicite=True,
            )
            print("Dépôt contrôlé :", preuve["depot"])
            print("SHA-256 vérifié :", preuve["sha256"])
            print("Statut :", preuve["verification"])
        except Exception as exc:
            print("Dépôt impossible :", type(exc).__name__, str(exc))
            print("Archive locale conservée :", livrable["archive"])
            print("Utiliser la cellule de secours ci-dessous si nécessaire.")
        finally:
            bouton_depot.disabled = False

# Ne pas déposer automatiquement comme « approuvée » une version à réviser.
if DECISION_SUITE == "reviser":
    confirmation_depot.disabled = True
    bouton_depot.disabled = True
    print("Texte à réviser : archive provisoire disponible dans la cellule suivante.")

bouton_depot.on_click(_deposer_td01)
display(widgets.VBox([confirmation_depot, bouton_depot, sortie_depot]))

In [ ]:
# En cas de révision : récupérer l'archive provisoire pour la suite.
# En cas d'échec du dépôt automatique : récupérer l'archive approuvée.
if EST_COLAB and DECISION_SUITE == "reviser":
    from google.colab import files
    files.download(livrable["archive"])
else:
    print("Archive locale disponible :", livrable["archive"])
    if EST_COLAB:
        print("Pour la télécharger manuellement :")
        print("from google.colab import files; files.download(livrable['archive'])")

## Pour poursuivre

La comparaison lexicale est disponible, quelle que soit votre décision sur le nettoyage.

- **Conserver** : le texte retenu et ses résultats pourront servir de base après revue collective.
- **Réviser** : conservez l'archive provisoire, corrigez les fonctions concernées, puis reprenez une copie du notebook avec une nouvelle `REVISION_GIT` et une nouvelle `VERSION_NOTEBOOK`. Les résultats précédents fournissent une référence de comparaison.

Le journal et la fiche de suivi permettent de retrouver précisément les conditions des calculs.